<a href="https://colab.research.google.com/github/Amebley/Bias-Mitigation/blob/main/Image_Labelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q deepface datasets pillow tqdm pandas

--- Demographic Counts in Validation Split ---

Most common AGE:
age
3    3300
4    2330
1    1356
5    1353
2    1181
6     796
7     321
0     199
8     118
Name: count, dtype: int64
Top: 3 (3300 occurrences)

Most common GENDER:
gender
0    5792
1    5162
Name: count, dtype: int64
Top: 0 (5792 occurrences)

Most common RACE:
race
3    2085
5    1623
2    1556
0    1550
1    1516
6    1415
4    1209
Name: count, dtype: int64
Top: 3 (2085 occurrences)


In [9]:
from datasets import load_dataset

# Load the full dataset
dataset = load_dataset("HuggingFaceM4/FairFace")

# Check what splits are available (train, validation, test)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image', 'age', 'gender', 'race', 'service_test'],
        num_rows: 86744
    })
    validation: Dataset({
        features: ['image', 'age', 'gender', 'race', 'service_test'],
        num_rows: 10954
    })
})


In [6]:
from deepface import DeepFace
import numpy as np
import cv2

# Quick test on one image to make sure everything works
sample_pil = dataset["train"][0]["image"]
sample_array = np.array(sample_pil)           # Convert PIL → NumPy (RGB)
sample_bgr = cv2.cvtColor(sample_array, cv2.COLOR_RGB2BGR)  # Convert RGB → BGR

result = DeepFace.analyze(
    img_path=sample_bgr,
    actions=["emotion"],
    enforce_detection=False,
    detector_backend="skip"   # Skip detection entirely
)

print(result[0]["dominant_emotion"])

26-10-28 16:25:31 - Directory /root/.deepface has been created
26-10-28 16:25:31 - Directory /root/.deepface/weights has been created
26-10-28 16:25:33 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 169MB/s]


neutral


In [10]:
from tqdm import tqdm
import pandas as pd

def get_emotion_labels(split_name):
    """Iterate over a dataset split and return a list of emotion labels."""
    ds = dataset[split_name]
    emotions = []

    for i in tqdm(range(len(ds)), desc=f"Labeling {split_name}"):
        pil_img = ds[i]["image"]
        img_array = np.array(pil_img)
        img_bgr = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)

        try:
            res = DeepFace.analyze(
                img_path=img_bgr,
                actions=["emotion"],
                enforce_detection=False,
                detector_backend="skip",
                silent=True   # Suppress DeepFace's internal logs
            )
            emotions.append(res[0]["dominant_emotion"])
        except Exception as e:
            # If anything fails, label as "unknown" instead of crashing
            emotions.append("unknown")

    return emotions

# Run for all three splits
train_emotions = get_emotion_labels("train")
val_emotions   = get_emotion_labels("validation")

Labeling validation: 100%|██████████| 10954/10954 [02:26<00:00, 74.88it/s]


KeyError: 'test'

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import pandas as pd
import os

# 1. Define your target save directory based on your previous path
save_dir = "/content/drive/MyDrive/Colab Notebooks/Implementation_of_AI_Project/Dataset/FairFace"

# Create the directory if it doesn't exist yet
os.makedirs(save_dir, exist_ok=True)

# 2. Convert Hugging Face splits to Pandas DataFrames
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()

# 3. Assign the generated emotion labels to the DataFrames
# We use direct assignment to avoid any duplicate column errors
train_df["emotion_label"] = train_emotions
val_df["emotion_label"] = val_emotions

# 4. Drop the 'image' column to keep the CSV files small and fast to read
train_df = train_df.drop(columns=["image"])
val_df = val_df.drop(columns=["image"])

# 5. Save the individual splits
train_path = os.path.join(save_dir, "train_labels.csv")
val_path = os.path.join(save_dir, "val_labels.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
print(f" Saved {train_path}")
print(f" Saved {val_path}")

# 6. Combine them and save the full dataset file
full_df = pd.concat([train_df, val_df], ignore_index=True)
full_path = os.path.join(save_dir, "full_data_with_emotions.csv")

full_df.to_csv(full_path, index=False)
print(f" Saved {full_path}")

print(f"\nTotal rows in full_data_with_emotions.csv: {len(full_df)}")

 Saved /content/drive/MyDrive/Colab Notebooks/Implementation_of_AI_Project/Dataset/FairFace/train_labels.csv
 Saved /content/drive/MyDrive/Colab Notebooks/Implementation_of_AI_Project/Dataset/FairFace/val_labels.csv
 Saved /content/drive/MyDrive/Colab Notebooks/Implementation_of_AI_Project/Dataset/FairFace/full_data_with_emotions.csv

Total rows in full_data_with_emotions.csv: 97698
